In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 60)

In [2]:
import torch
from transformers import T5Tokenizer, T5EncoderModel
from tqdm.auto import tqdm

device = "mps" if torch.backends.mps.is_available() else \
         ("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load ProstT5 (first run downloads ~1.1 GB; cached after)
print("Loading ProstT5 (Rostlab/ProstT5) ...")
tokenizer = T5Tokenizer.from_pretrained('Rostlab/ProstT5', do_lower_case=False)
model = T5EncoderModel.from_pretrained("Rostlab/ProstT5", torch_dtype=torch.float32)
model.eval().to(device)
print(f"Loaded. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# Load sequences and DisProt disorder regions
seqs = pd.read_csv("../data/sequences.csv")
disprot = pd.read_csv("../data/disprot.tsv", sep='\t', low_memory=False)
disprot = disprot.rename(columns={'UniProt ACC':'acc', 'Organism':'organism',
                                  'Term namespace':'term_namespace',
                                  'Start':'start', 'End':'end'})
disprot['acc'] = disprot['acc'].str.split('-').str[0]
state = disprot[(disprot['organism']=='Homo sapiens') &
                (disprot['term_namespace']=='Structural state')]
disorder_regions = (state.groupby('acc')
                         .apply(lambda x: [(int(s), int(e)) for s, e in zip(x['start'], x['end'])])
                         .to_dict())

MAX_LEN = 1022  # ProstT5's practical context, same as ESM-2

@torch.no_grad()
def embed_disorder_prostt5(seq, acc):
    s = seq[:MAX_LEN].upper()
    # ProstT5: replace unusual AAs with X (per Rostlab guidance)
    s = ''.join(['X' if c in 'UZOB' else c for c in s])
    # ProstT5 expects space-separated AAs with <AA2fold> prefix for AA-only mode
    prepared = "<AA2fold> " + " ".join(list(s))

    tokens = tokenizer(prepared, add_special_tokens=True,
                       return_tensors='pt')
    ids = tokens['input_ids'].to(device)
    mask = tokens['attention_mask'].to(device)

    out = model(input_ids=ids, attention_mask=mask)
    # last_hidden_state shape: (1, seq_len_tokens, 1024)
    # Layout: <AA2fold>, aa_1, aa_2, ..., aa_L, </s>
    # Skip prefix token (index 0) and terminal </s> (last index)
    per_residue = out.last_hidden_state[0, 1:1+len(s)]  # shape (L, 1024)
    L = per_residue.shape[0]

    regions = disorder_regions.get(acc, [])
    if not regions:
        return per_residue.mean(dim=0).cpu().numpy().astype("float32"), True

    indices = []
    for start, end in regions:
        for i in range(start-1, min(end, L)):
            indices.append(i)

    if not indices:
        return per_residue.mean(dim=0).cpu().numpy().astype("float32"), True

    idx_t = torch.tensor(indices, device=device, dtype=torch.long)
    selected = per_residue[idx_t]
    return selected.mean(dim=0).cpu().numpy().astype("float32"), False

embeddings = {}
fallbacks = []
for _, row in tqdm(seqs.iterrows(), total=len(seqs)):
    try:
        emb, fb = embed_disorder_prostt5(row['sequence'], row['acc'])
        embeddings[row['acc']] = emb
        if fb:
            fallbacks.append(row['acc'])
    except Exception as e:
        print(f"  Failed for {row['acc']}: {type(e).__name__}: {e}")

accs = list(embeddings.keys())
X = np.stack([embeddings[a] for a in accs])
np.savez_compressed("../data/features_prostt5_disorder.npz",
                    accs=np.array(accs), X=X.astype("float32"))
print(f"\nSaved {X.shape[0]} disorder-pooled ProstT5 embeddings of dim {X.shape[1]}")
print(f"Fallback (no disorder in truncation window): {len(fallbacks)} proteins")

Using device: mps
Loading ProstT5 (Rostlab/ProstT5) ...


tokenizer_config.json:   0%|          | 0.00/2.60k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/11.3G [00:00<?, ?B/s]

Loaded. Parameters: 1208.2M


  0%|          | 0/1279 [00:00<?, ?it/s]


Saved 1279 disorder-pooled ProstT5 embeddings of dim 1024
Fallback (no disorder in truncation window): 41 proteins


In [3]:
data = np.load("../data/features_prostt5_disorder.npz", allow_pickle=True)
X, accs = data['X'], data['accs']
norms = np.linalg.norm(X, axis=1)
print("shape:", X.shape, "dtype:", X.dtype)
print(f"L2 norms: min={norms.min():.2f}, max={norms.max():.2f}, mean={norms.mean():.2f}")
print(f"NaNs? {np.isnan(X).any()}; zero-norm vectors? {(norms==0).sum()}")

shape: (1279, 1024) dtype: float32
L2 norms: min=1.85, max=6.54, mean=4.01
NaNs? False; zero-norm vectors? 0
